# PlasticWise AI - 02_features_and_baselines
**Week 3:** image pipeline, augmentation figure, MobileNetV2 embeddings + t-SNE, Logistic Regression / Random Forest baselines, frozen MobileNetV2 model.

**Pehle:** Runtime > Change runtime type > **T4 GPU**. Cells ko upar se neeche ek ek karke run karo. Week 1-2 ka kaam Drive mein save hona chahiye (`PlasticWise/data/train.csv` wagera).

Is notebook ke sab outputs Drive ke `PlasticWise/results/week3/` aur `models/` mein save hote hain.

## Cell 1: Setup (Drive + dataset dobara)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
from pathlib import Path

BASE = '/content/drive/MyDrive/PlasticWise'
os.makedirs(f'{BASE}/results/week3', exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)

import kagglehub
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_data_dir(root):
    best, best_n = None, 0
    for d in [Path(root)] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        classes = [s for s in d.iterdir() if s.is_dir() and any(f.suffix.lower() in IMG_EXT for f in s.iterdir() if f.is_file())]
        if len(classes) > best_n:
            best, best_n = d, len(classes)
    return best

DATA_DIR = find_data_dir(path)
print("DATA_DIR =", DATA_DIR)

## Cell 2: Settings, CSVs aur class weights load karo

In [ ]:
import numpy as np, pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
IMG_SIZE = 224
BATCH = 32
WEIGHTS = 'imagenet'      # ImageNet pretrained weights
EPOCHS = 10
tf.keras.utils.set_random_seed(SEED)

train = pd.read_csv(f'{BASE}/data/train.csv')
val   = pd.read_csv(f'{BASE}/data/val.csv')
test  = pd.read_csv(f'{BASE}/data/test.csv')

# Agar session naya hone se file paths badal gaye hon to relpath se dobara banao
for d in (train, val, test):
    if not Path(d.loc[0, 'filepath']).exists():
        d['filepath'] = d['relpath'].apply(lambda r: str(DATA_DIR / r))

class_names = json.load(open(f'{BASE}/data/class_names.json'))
class_weights = {int(k): v for k, v in json.load(open(f'{BASE}/data/class_weights.json')).items()}
NUM_CLASSES = len(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}

y_train = train['label'].map(class_to_idx).values
y_val   = val['label'].map(class_to_idx).values
y_test  = test['label'].map(class_to_idx).values

print(f"train {len(train)} | val {len(val)} | test {len(test)} | classes {NUM_CLASSES}")
print("File exists check:", Path(train.loc[0, 'filepath']).exists())

## Cell 3: tf.data input pipeline
Memory safe tareeqa: images ko 224x224 **uint8** (chhoti memory) mein cache karte hain, augmentation aur `preprocess_input` baad mein batch par lagta hai. Augmentation **sirf training** par hoti hai.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

def load_and_resize(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8)

def make_base_ds(df):
    # Decode + resize + cache (aik hi dafa). Order df jaisa rehta hai.
    return (tf.data.Dataset.from_tensor_slices(df['filepath'].values)
            .map(load_and_resize, num_parallel_calls=AUTOTUNE)
            .cache())

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),     # value_range default (0, 255)
], name='augmentation')

def prep_only(x):
    return preprocess_input(tf.cast(x, tf.float32))

def prep_aug(x):
    return preprocess_input(augment(tf.cast(x, tf.float32), training=True))

base_train = make_base_ds(train)
base_val   = make_base_ds(val)
base_test  = make_base_ds(test)

def labels_ds(y):
    return tf.data.Dataset.from_tensor_slices(tf.one_hot(y, NUM_CLASSES))

# Embeddings ke liye (augmentation nahi, shuffle nahi)
emb_train = base_train.batch(BATCH).map(prep_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
emb_val   = base_val.batch(BATCH).map(prep_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
emb_test  = base_test.batch(BATCH).map(prep_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Model training ke liye (train par augmentation + shuffle, one-hot labels)
train_ds = (tf.data.Dataset.zip((base_train, labels_ds(y_train)))
            .shuffle(2000, seed=SEED).batch(BATCH)
            .map(lambda x, y: (prep_aug(x), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds   = (tf.data.Dataset.zip((base_val, labels_ds(y_val))).batch(BATCH)
            .map(lambda x, y: (prep_only(x), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
test_ds  = (tf.data.Dataset.zip((base_test, labels_ds(y_test))).batch(BATCH)
            .map(lambda x, y: (prep_only(x), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
print("Pipeline ready.")

## Cell 4: Augmentation se pehle aur baad (report figure)

In [ ]:
import matplotlib.pyplot as plt

sample_row = train[train['label'] == 'plastic'].iloc[0]
orig = load_and_resize(sample_row['filepath'])          # uint8 224x224x3

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(orig.numpy()); axes[0, 0].set_title('ORIGINAL (224x224)'); axes[0, 0].axis('off')
slots = [a for a in axes.ravel()][1:]
for ax in slots:
    a = augment(tf.cast(orig, tf.float32)[tf.newaxis], training=True)[0]
    ax.imshow(np.clip(a.numpy(), 0, 255).astype('uint8'))
    ax.set_title('Augmented'); ax.axis('off')
plt.suptitle(f"Augmentation example (class: {sample_row['label']})", fontsize=14)
plt.tight_layout()
plt.savefig(f'{BASE}/results/week3/augmentation_example.png', dpi=200)
plt.show()

### Feature engineering (images ke liye) - 5 lines
1. Raw image ke pixels ko seedha model ko dene ke bajaye hum unhein 224x224 par resize karke ek standard shape dete hain.
2. `preprocess_input` pixel values ko -1 se 1 ki range mein laata hai, jo MobileNetV2 ki training jaisi hai.
3. Augmentation (flip, rotation, zoom, contrast, brightness) naye training variations banata hai, isse overfitting kam hoti hai.
4. Frozen MobileNetV2 har image se 1280 numbers ka **embedding** nikalta hai. Ye automatic "learned features" hain.
5. Un embeddings par Logistic Regression aur Random Forest jaise simple ML models chalte hain, yani CNN features + classical ML.

## Cell 5: MobileNetV2 (frozen) se 1280-dim embeddings nikalo
GPU par 12k images ke liye 1-3 minute. Embeddings Drive mein `.npy` file mein save hote hain.

In [ ]:
feat_extractor = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
    weights=WEIGHTS, pooling='avg')
feat_extractor.trainable = False

X_train = feat_extractor.predict(emb_train, verbose=1)
X_val   = feat_extractor.predict(emb_val, verbose=1)
X_test  = feat_extractor.predict(emb_test, verbose=1)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                  ('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    np.save(f'{BASE}/results/week3/{name}.npy', arr)
print("Saved embeddings (.npy) to Drive.")

## Cell 6: PCA aur t-SNE plot (class ke rang mein)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

n = min(3000, len(X_train))
idx = np.random.default_rng(SEED).choice(len(X_train), n, replace=False)
Xs, ys = X_train[idx], y_train[idx]

pca2 = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
pre = PCA(n_components=min(50, Xs.shape[1], n - 1), random_state=SEED).fit_transform(Xs)
tsne2 = TSNE(n_components=2, perplexity=min(30, (n - 1) / 3), init='pca',
             learning_rate='auto', random_state=SEED).fit_transform(pre)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cmap = plt.get_cmap('tab10')
for ax, pts, title in [(axes[0], pca2, 'PCA (2D)'), (axes[1], tsne2, 't-SNE (2D)')]:
    for i, c in enumerate(class_names):
        m = ys == i
        ax.scatter(pts[m, 0], pts[m, 1], s=8, alpha=0.7, color=cmap(i % 10), label=c)
    ax.set_title(f'{title} of MobileNetV2 embeddings')
axes[1].legend(markerscale=3, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{BASE}/results/week3/tsne_pca_embeddings.png', dpi=200)
plt.show()

## Cell 7: Baselines - Logistic Regression aur Random Forest (embeddings par)

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

results = []

t0 = time.time()
lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=1000, class_weight='balanced'))
lr.fit(X_train, y_train)
pred = lr.predict(X_val)
results.append({'model': 'Logistic Regression (embeddings)',
                'val_accuracy': accuracy_score(y_val, pred),
                'val_macro_f1': f1_score(y_val, pred, average='macro')})
print(f"Logistic Regression done in {time.time()-t0:.0f}s ->", results[-1])

t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                            n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)
pred = rf.predict(X_val)
results.append({'model': 'Random Forest (embeddings)',
                'val_accuracy': accuracy_score(y_val, pred),
                'val_macro_f1': f1_score(y_val, pred, average='macro')})
print(f"Random Forest done in {time.time()-t0:.0f}s ->", results[-1])

## Cell 8: Frozen MobileNetV2 model train karo
Base frozen hai, sirf upar wali nayi layers seekhti hain. 10 epochs (EarlyStopping pehle bhi rok sakti hai).

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=WEIGHTS)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs, name='plasticwise_frozen')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

CKPT = f'{BASE}/models/mobilenetv2_frozen_best.keras'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(CKPT, monitor='val_loss', save_best_only=True),
    CSVLogger(f'{BASE}/results/week3/frozen_training_log.csv'),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weights, callbacks=callbacks)

## Cell 9: Accuracy / loss curves

In [ ]:
h = history.history
ep = range(1, len(h['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(ep, h['accuracy'], label='train'); axes[0].plot(ep, h['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(ep, h['loss'], label='train'); axes[1].plot(ep, h['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{BASE}/results/week3/frozen_curves.png', dpi=200)
plt.show()

## Cell 10: Teeno models ka results table
`restore_best_weights=True` ki wajah se model best epoch ke weights par hai.

In [ ]:
probs = model.predict(val_ds, verbose=0)
pred = probs.argmax(axis=1)
results.append({'model': 'Frozen MobileNetV2 (transfer learning)',
                'val_accuracy': accuracy_score(y_val, pred),
                'val_macro_f1': f1_score(y_val, pred, average='macro')})

res = pd.DataFrame(results)
res['val_accuracy'] = res['val_accuracy'].round(4)
res['val_macro_f1'] = res['val_macro_f1'].round(4)
print(res.to_string(index=False))
res.to_csv(f'{BASE}/results/week3/week3_results.csv', index=False)

ax = res.set_index('model')[['val_accuracy', 'val_macro_f1']].plot.bar(figsize=(9, 5), rot=15)
ax.set_title('Week 3: validation accuracy and macro-F1'); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(f'{BASE}/results/week3/week3_results.png', dpi=200); plt.show()
print("Saved: week3_results.csv / .png")

**Note:** Test set ko abhi use nahi kiya. Test sirf aakhri model par **ek hi dafa** (Week 4-5) chalana hai, warna result bias ho jata hai.

## Week 3 checklist
- [ ] `augmentation_example.png` bani
- [ ] `tsne_pca_embeddings.png` bani
- [ ] Embeddings `.npy` Drive mein save hue
- [ ] `week3_results.csv` mein 3 models ka table hai
- [ ] `models/mobilenetv2_frozen_best.keras` Drive mein hai
- [ ] Notebook GitHub par `notebooks/` mein commit